In [1]:
import pandas as pd
import numpy as np
import joblib

from pathlib import Path
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

BASE_DIR = Path.cwd().parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"

In [2]:
xgb_champion = joblib.load(
    BASE_DIR / "notebooks" / "xgb_day7_champion.pkl"
)

X_validation = __import__("scipy").sparse.load_npz(
    PROCESSED_DIR / "X_validation.npz"
)

y_validation = pd.read_csv(
    PROCESSED_DIR / "y_validation.csv"
)["isFraud"]

print("Champion loaded successfully.")
print("Validation shape:", X_validation.shape)
print("Validation target:", y_validation.shape)

Champion loaded successfully.
Validation shape: (105693, 891)
Validation target: (105693,)


In [3]:
y_day8_proba = xgb_champion.predict_proba(
    X_validation
)[:, 1]

print(
    "ROC-AUC:",
    round(
        roc_auc_score(y_validation, y_day8_proba),
        4
    )
)

print(
    "PR-AUC:",
    round(
        average_precision_score(y_validation, y_day8_proba),
        4
    )
)

ROC-AUC: 0.9219
PR-AUC: 0.6031


In [4]:
final_threshold = 0.20

day8_results = pd.DataFrame({
    "actual": y_validation.values,
    "probability": y_day8_proba
})

day8_results["prediction"] = (
    day8_results["probability"] >= final_threshold
).astype(int)

day8_results["error_type"] = np.select(
    [
        (day8_results["actual"] == 0) &
        (day8_results["prediction"] == 0),

        (day8_results["actual"] == 0) &
        (day8_results["prediction"] == 1),

        (day8_results["actual"] == 1) &
        (day8_results["prediction"] == 0),

        (day8_results["actual"] == 1) &
        (day8_results["prediction"] == 1)
    ],
    [
        "True Negative",
        "False Positive",
        "False Negative",
        "True Positive"
    ],
    default="Unknown"
)

print(
    day8_results["error_type"].value_counts()
)

error_type
True Negative     101386
False Negative      1833
True Positive       1778
False Positive       696
Name: count, dtype: int64


In [5]:
false_negatives = day8_results[
    day8_results["error_type"] == "False Negative"
].copy()

print("False negatives:", len(false_negatives))

print(
    false_negatives["probability"].describe()
)

False negatives: 1833
count    1833.000000
mean        0.036936
std         0.047469
min         0.000018
25%         0.003125
50%         0.014637
75%         0.055088
max         0.199977
Name: probability, dtype: float64


In [6]:
false_negatives["probability_bin"] = pd.cut(
    false_negatives["probability"],
    bins=[
        -0.001,
        0.05,
        0.10,
        0.15,
        0.20
    ],
    labels=[
        "0-0.05",
        "0.05-0.10",
        "0.10-0.15",
        "0.15-0.20"
    ],
    include_lowest=True
)

display(
    false_negatives["probability_bin"]
    .value_counts()
    .sort_index()
)

probability_bin
0-0.05       1337
0.05-0.10     262
0.10-0.15     155
0.15-0.20      79
Name: count, dtype: int64

In [9]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd().parent

DATA_RAW = BASE_DIR / "data" / "raw"
DATA_PROCESSED = BASE_DIR / "data" / "processed"

print("BASE_DIR:", BASE_DIR)
print("Raw exists:", DATA_RAW.exists())
print("Processed exists:", DATA_PROCESSED.exists())

BASE_DIR: /Users/ayushkumar/Desktop/Fraudguard
Raw exists: True
Processed exists: True


In [10]:
print(list(DATA_PROCESSED.iterdir()))

[PosixPath('/Users/ayushkumar/Desktop/Fraudguard/data/processed/X_validation.npz'), PosixPath('/Users/ayushkumar/Desktop/Fraudguard/data/processed/feature_names.csv'), PosixPath('/Users/ayushkumar/Desktop/Fraudguard/data/processed/X_train.npz'), PosixPath('/Users/ayushkumar/Desktop/Fraudguard/data/processed/xgboost_fraudguard.pkl'), PosixPath('/Users/ayushkumar/Desktop/Fraudguard/data/processed/xgboost_feature_importance.csv'), PosixPath('/Users/ayushkumar/Desktop/Fraudguard/data/processed/y_validation.csv'), PosixPath('/Users/ayushkumar/Desktop/Fraudguard/data/processed/y_train.csv')]


In [19]:
train = train_transaction.copy()

train["transaction_day"] = (
    train["TransactionDT"] // (24 * 60 * 60)
)

In [20]:
train_data = train[
    train["transaction_day"] <= 145
].copy()

validation_data = train[
    train["transaction_day"] > 145
].copy()

print("Training:", train_data.shape)
print("Validation:", validation_data.shape)

Training: (484847, 395)
Validation: (105693, 395)


In [21]:
print(
    "Target alignment:",
    np.array_equal(
        validation_data["isFraud"].values,
        y_validation.values
    )
)

Target alignment: True


In [23]:
fn_analysis = validation_data.iloc[
    false_negatives.index
].copy()

fn_analysis["model_probability"] = (
    false_negatives["probability"].values
)

print("False-negative rows:", len(fn_analysis))

False-negative rows: 1833


In [24]:
display(
    fn_analysis[
        [
            "TransactionID",
            "TransactionDT",
            "isFraud",
            "V258",
            "V294",
            "TransactionAmt",
            "model_probability"
        ]
    ].head(20)
)

,TransactionID,TransactionDT,isFraud,V258,V294,TransactionAmt,model_probability
484851,3471851,12614510,1,1.0,2.0,19.059,0.076578
484856,3471856,12614560,1,1.0,3.0,19.059,0.054494
484872,3471872,12614979,1,NaN,0.0,311.950,0.065398
484914,3471914,12615642,1,NaN,0.0,77.000,0.009844
484922,3471922,12615934,1,NaN,0.0,269.950,0.003842
485068,3472068,12619206,1,1.0,0.0,200.000,0.054378
485073,3472073,12619330,1,1.0,0.0,200.000,0.025557
485127,3472127,12620807,1,NaN,0.0,550.490,0.002415
485211,3472211,12623306,1,NaN,0.0,226.000,0.017200
485229,3472229,12623629,1,NaN,1.0,226.000,0.014492


In [26]:
tp_indices = day8_results[
    day8_results["error_type"] == "True Positive"
].index

tp_analysis = validation_data.iloc[
    tp_indices
].copy()

print("True-positive rows:", len(tp_analysis))

display(
    tp_analysis["V258"].describe()
)

True-positive rows: 1778


count    1301.000000
mean        3.981553
std         3.852228
min         1.000000
25%         1.000000
50%         3.000000
75%         5.000000
max        24.000000
Name: V258, dtype: float64

In [27]:
print("V258 missingness comparison")
print("--------------------------------")

print(
    "False Negative missing:",
    fn_analysis["V258"].isna().sum(),
    "/",
    len(fn_analysis),
    f"({fn_analysis['V258'].isna().mean() * 100:.2f}%)"
)

print(
    "True Positive missing:",
    tp_analysis["V258"].isna().sum(),
    "/",
    len(tp_analysis),
    f"({tp_analysis['V258'].isna().mean() * 100:.2f}%)"
)

V258 missingness comparison
--------------------------------
False Negative missing: 1285 / 1833 (70.10%)
True Positive missing: 477 / 1778 (26.83%)


In [28]:
print("V258 value distribution — False Negatives")
display(
    fn_analysis["V258"]
    .value_counts(dropna=False)
    .head(15)
)

print("\nV258 value distribution — True Positives")
display(
    tp_analysis["V258"]
    .value_counts(dropna=False)
    .head(15)
)

V258 value distribution — False Negatives


V258
NaN     1285
1.0      403
2.0       83
3.0       52
5.0        4
4.0        3
13.0       2
6.0        1
Name: count, dtype: int64


V258 value distribution — True Positives


V258
NaN     477
1.0     359
2.0     262
3.0     169
4.0     142
5.0      94
6.0      61
7.0      45
8.0      30
9.0      25
11.0     19
10.0     17
13.0     15
12.0     11
17.0      9
Name: count, dtype: int64

In [29]:
print("V258 missingness by model outcome")
print("----------------------------------")

print(
    "False Negative:",
    fn_analysis["V258"].isna().mean() * 100
)

print(
    "True Positive:",
    tp_analysis["V258"].isna().mean() * 100
)

V258 missingness by model outcome
----------------------------------
False Negative: 70.10365521003818
True Positive: 26.827896512935883


In [30]:
tn_indices = day8_results[
    day8_results["error_type"] == "True Negative"
].index

fp_indices = day8_results[
    day8_results["error_type"] == "False Positive"
].index

tn_analysis = validation_data.iloc[tn_indices].copy()
fp_analysis = validation_data.iloc[fp_indices].copy()

In [31]:
print(
    "True Negative V258 missing:",
    tn_analysis["V258"].isna().mean() * 100
)

print(
    "False Positive V258 missing:",
    fp_analysis["V258"].isna().mean() * 100
)

True Negative V258 missing: 83.87449943779221
False Positive V258 missing: 31.178160919540232


In [32]:
fn_v258_missing = fn_analysis[
    fn_analysis["V258"].isna()
].copy()

tn_v258_missing = tn_analysis[
    tn_analysis["V258"].isna()
].copy()

print("FN with V258 missing:", len(fn_v258_missing))
print("TN with V258 missing:", len(tn_v258_missing))

FN with V258 missing: 1285
TN with V258 missing: 85037


In [33]:
comparison_features = [
    "TransactionAmt",
    "TransactionDT",
    "V294",
    "card1",
    "card2",
    "card3",
    "card4",
    "card5",
    "card6",
    "ProductCD",
    "addr1",
    "addr2"
]

comparison = []

for feature in comparison_features:

    if feature in validation_data.columns:

        fn_missing = fn_v258_missing[feature].isna().mean() * 100
        tn_missing = tn_v258_missing[feature].isna().mean() * 100

        comparison.append({
            "feature": feature,
            "FN_missing_%": round(fn_missing, 2),
            "TN_missing_%": round(tn_missing, 2),
            "difference": round(
                fn_missing - tn_missing,
                2
            )
        })

comparison_df = pd.DataFrame(comparison)

display(
    comparison_df.sort_values(
        "difference",
        ascending=False
    )
)

,feature,FN_missing_%,TN_missing_%,difference
10,addr1,5.29,2.08,3.21
11,addr2,5.29,2.08,3.21
0,TransactionAmt,0.00,0.00,0.00
1,TransactionDT,0.00,0.00,0.00
2,V294,0.00,0.00,0.00
3,card1,0.00,0.00,0.00
9,ProductCD,0.00,0.00,0.00
7,card5,0.78,0.99,-0.22
5,card3,0.31,0.75,-0.44
6,card4,0.31,0.75,-0.44


In [34]:
fn_v258_missing["addr_missing"] = (
    fn_v258_missing["addr1"].isna() &
    fn_v258_missing["addr2"].isna()
)

tn_v258_missing["addr_missing"] = (
    tn_v258_missing["addr1"].isna() &
    tn_v258_missing["addr2"].isna()
)

print(
    "FN: V258 missing + addr missing:",
    fn_v258_missing["addr_missing"].sum(),
    f"({fn_v258_missing['addr_missing'].mean() * 100:.2f}%)"
)

print(
    "TN: V258 missing + addr missing:",
    tn_v258_missing["addr_missing"].sum(),
    f"({tn_v258_missing['addr_missing'].mean() * 100:.2f}%)"
)

FN: V258 missing + addr missing: 68 (5.29%)
TN: V258 missing + addr missing: 1771 (2.08%)


In [35]:
fn_v258_missing["addr_any_missing"] = (
    fn_v258_missing["addr1"].isna() |
    fn_v258_missing["addr2"].isna()
)

tn_v258_missing["addr_any_missing"] = (
    tn_v258_missing["addr1"].isna() |
    tn_v258_missing["addr2"].isna()
)

print(
    "FN: V258 missing + any addr missing:",
    fn_v258_missing["addr_any_missing"].mean() * 100
)

print(
    "TN: V258 missing + any addr missing:",
    tn_v258_missing["addr_any_missing"].mean() * 100
)

FN: V258 missing + any addr missing: 5.291828793774319
TN: V258 missing + any addr missing: 2.0826228582852173


In [36]:
all_features = [
    col for col in validation_data.columns
    if col not in ["isFraud", "TransactionID"]
]

feature_comparison = []

for feature in all_features:

    fn_missing = fn_v258_missing[feature].isna().mean()
    tn_missing = tn_v258_missing[feature].isna().mean()

    feature_comparison.append({
        "feature": feature,
        "FN_missing_%": fn_missing * 100,
        "TN_missing_%": tn_missing * 100,
        "absolute_difference": abs(
            fn_missing - tn_missing
        ) * 100
    })

feature_comparison_df = pd.DataFrame(
    feature_comparison
)

display(
    feature_comparison_df.sort_values(
        "absolute_difference",
        ascending=False
    ).head(30)
)

,feature,FN_missing_%,TN_missing_%,absolute_difference
47,M4,18.443580,44.959253,26.515673
48,M5,24.124514,47.893270,23.768756
11,dist1,52.840467,45.676588,7.163879
30,D2,40.622568,35.247010,5.375558
33,D5,45.603113,41.866482,3.736631
10,addr2,5.291829,2.082623,3.209206
9,addr1,5.291829,2.082623,3.209206
54,V2,15.719844,12.686242,3.033602
55,V3,15.719844,12.686242,3.033602
63,V11,15.719844,12.686242,3.033602


In [37]:
print("M4 — Hard False Negatives")
display(
    fn_v258_missing["M4"]
    .value_counts(dropna=False, normalize=True)
    .mul(100)
    .round(2)
)

print("\nM4 — True Negatives")
display(
    tn_v258_missing["M4"]
    .value_counts(dropna=False, normalize=True)
    .mul(100)
    .round(2)
)

M4 — Hard False Negatives


M4
M0     59.22
NaN    18.44
M1     14.16
M2      8.17
Name: proportion, dtype: float64


M4 — True Negatives


M4
NaN    44.96
M0     40.62
M1     11.54
M2      2.88
Name: proportion, dtype: float64

In [38]:
print("M5 — Hard False Negatives")
display(
    fn_v258_missing["M5"]
    .value_counts(dropna=False, normalize=True)
    .mul(100)
    .round(2)
)

print("\nM5 — True Negatives")
display(
    tn_v258_missing["M5"]
    .value_counts(dropna=False, normalize=True)
    .mul(100)
    .round(2)
)

M5 — Hard False Negatives


M5
T      39.22
F      36.65
NaN    24.12
Name: proportion, dtype: float64


M5 — True Negatives


M5
NaN    47.89
F      28.74
T      23.37
Name: proportion, dtype: float64

In [39]:
print("V258 missing + M4=M0:")
print(
    fn_v258_missing["M4"].eq("M0").mean() * 100
)

print(
    tn_v258_missing["M4"].eq("M0").mean() * 100
)

print("\nV258 missing + M5=T:")
print(
    fn_v258_missing["M5"].eq("T").mean() * 100
)

print(
    tn_v258_missing["M5"].eq("T").mean() * 100
)

V258 missing + M4=M0:
59.221789883268485
40.61996542681421

V258 missing + M5=T:
39.221789883268485
23.36629937556593


In [40]:
fn_combined = (
    fn_v258_missing["M4"].eq("M0") &
    fn_v258_missing["M5"].eq("T")
)

tn_combined = (
    tn_v258_missing["M4"].eq("M0") &
    tn_v258_missing["M5"].eq("T")
)

print("\nV258 missing + M4=M0 + M5=T")
print(
    "FN:",
    fn_combined.sum(),
    f"({fn_combined.mean() * 100:.2f}%)"
)

print(
    "TN:",
    tn_combined.sum(),
    f"({tn_combined.mean() * 100:.2f}%)"
)


V258 missing + M4=M0 + M5=T
FN: 366 (28.48%)
TN: 15328 (18.03%)


In [41]:
train_data["v258_m4_m5_interaction"] = (
    train_data["V258"].isna() &
    train_data["M4"].eq("M0") &
    train_data["M5"].eq("T")
).astype(np.int8)

validation_data["v258_m4_m5_interaction"] = (
    validation_data["V258"].isna() &
    validation_data["M4"].eq("M0") &
    validation_data["M5"].eq("T")
).astype(np.int8)

print(
    "Training interaction count:",
    train_data["v258_m4_m5_interaction"].sum()
)

print(
    "Validation interaction count:",
    validation_data["v258_m4_m5_interaction"].sum()
)

Training interaction count: 67267
Validation interaction count: 15847


In [42]:
print(
    "Training interaction rate:",
    train_data["v258_m4_m5_interaction"].mean() * 100
)

print(
    "Validation interaction rate:",
    validation_data["v258_m4_m5_interaction"].mean() * 100
)

Training interaction rate: 13.873861238700044
Validation interaction rate: 14.993424351659995


In [44]:
# Recreate feature lists from the current training data

TARGET = "isFraud"
ID_COLUMN = "TransactionID"

feature_columns = [
    col for col in train_data.columns
    if col not in [TARGET, ID_COLUMN]
]

numerical_features = train_data[
    feature_columns
].select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = train_data[
    feature_columns
].select_dtypes(
    exclude=["number"]
).columns.tolist()

print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))
print("Total:", len(feature_columns))

Numerical features: 380
Categorical features: 14
Total: 394


In [45]:
numerical_features_exp1 = numerical_features.copy()

print(
    "Numerical features:",
    len(numerical_features_exp1)
)

print(
    "Interaction present:",
    "v258_m4_m5_interaction" in numerical_features_exp1
)

Numerical features: 380
Interaction present: True


In [47]:
from sklearn.impute import SimpleImputer
from scipy.sparse import hstack

In [48]:
numerical_imputer_exp1 = SimpleImputer(
    strategy="median"
)

X_train_num_exp1 = numerical_imputer_exp1.fit_transform(
    train_data[numerical_features]
)

X_validation_num_exp1 = numerical_imputer_exp1.transform(
    validation_data[numerical_features]
)

print(
    "Training numerical matrix:",
    X_train_num_exp1.shape
)

print(
    "Validation numerical matrix:",
    X_validation_num_exp1.shape
)

Training numerical matrix: (484847, 380)
Validation numerical matrix: (105693, 380)


In [50]:
from sklearn.preprocessing import OneHotEncoder

categorical_features = train_data[
    [
        col for col in train_data.columns
        if col not in ["isFraud", "TransactionID"]
    ]
].select_dtypes(exclude=["number"]).columns.tolist()

print("Categorical features:", len(categorical_features))

Categorical features: 14


In [51]:
print("transform_categorical exists:", "transform_categorical" in globals())
print("categorical_maps exists:", "categorical_maps" in globals())

transform_categorical exists: False
categorical_maps exists: False


In [54]:
TARGET = "isFraud"
ID_COLUMN = "TransactionID"

X_train = train_data.drop(
    columns=[TARGET, ID_COLUMN]
)

y_train = train_data[TARGET]

X_validation = validation_data.drop(
    columns=[TARGET, ID_COLUMN]
)

y_validation = validation_data[TARGET]

print("X_train:", X_train.shape)
print("X_validation:", X_validation.shape)
print("y_train:", y_train.shape)
print("y_validation:", y_validation.shape)

X_train: (484847, 394)
X_validation: (105693, 394)
y_train: (484847,)
y_validation: (105693,)


In [55]:
RARE_THRESHOLD = 50

categorical_maps = {}

for feature in categorical_features:
    values = X_train[feature].fillna("__MISSING__")
    
    counts = values.value_counts()
    
    frequent_categories = counts[
        counts >= RARE_THRESHOLD
    ].index
    
    categorical_maps[feature] = set(
        frequent_categories
    )

print("Categorical mappings created:", len(categorical_maps))

Categorical mappings created: 14


In [57]:
def transform_categorical(
    df,
    categorical_columns,
    categorical_maps
):
    df = df[categorical_columns].copy()

    for feature in categorical_columns:
        df[feature] = df[feature].fillna("__MISSING__")

        frequent_categories = categorical_maps[feature]

        df[feature] = df[feature].where(
            df[feature].isin(frequent_categories),
            "__RARE__"
        )

    return df

In [58]:
X_train_cat = transform_categorical(
    X_train,
    categorical_features,
    categorical_maps
)

X_validation_cat = transform_categorical(
    X_validation,
    categorical_features,
    categorical_maps
)

print("Training categorical shape:", X_train_cat.shape)
print("Validation categorical shape:", X_validation_cat.shape)

Training categorical shape: (484847, 14)
Validation categorical shape: (105693, 14)


In [59]:
from sklearn.preprocessing import OneHotEncoder

categorical_encoder_exp1 = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True,
    dtype=np.float32
)

X_train_cat_encoded_exp1 = categorical_encoder_exp1.fit_transform(
    X_train_cat
)

X_validation_cat_encoded_exp1 = categorical_encoder_exp1.transform(
    X_validation_cat
)

print(
    "Encoded training shape:",
    X_train_cat_encoded_exp1.shape
)

print(
    "Encoded validation shape:",
    X_validation_cat_encoded_exp1.shape
)

Encoded training shape: (484847, 139)
Encoded validation shape: (105693, 139)


In [60]:
from scipy.sparse import hstack

X_train_exp1 = hstack([
    X_train_num_exp1,
    X_train_cat_encoded_exp1
]).tocsr()

X_validation_exp1 = hstack([
    X_validation_num_exp1,
    X_validation_cat_encoded_exp1
]).tocsr()

print("Day 8 Training shape:", X_train_exp1.shape)
print("Day 8 Validation shape:", X_validation_exp1.shape)

print("Training target:", y_train.shape)
print("Validation target:", y_validation.shape)

Day 8 Training shape: (484847, 519)
Day 8 Validation shape: (105693, 519)
Training target: (484847,)
Validation target: (105693,)


In [61]:
from scipy.sparse import load_npz

X_train_day7 = load_npz(
    "../data/processed/X_train.npz"
)

X_validation_day7 = load_npz(
    "../data/processed/X_validation.npz"
)

print("Day 7 training:", X_train_day7.shape)
print("Day 7 validation:", X_validation_day7.shape)

Day 7 training: (484847, 891)
Day 7 validation: (105693, 891)


In [62]:
X_train_day7 = load_npz(
    "/Users/ayushkumar/Desktop/Fraudguard/data/processed/X_train.npz"
)

X_validation_day7 = load_npz(
    "/Users/ayushkumar/Desktop/Fraudguard/data/processed/X_validation.npz"
)

print("Day 7 training:", X_train_day7.shape)
print("Day 7 validation:", X_validation_day7.shape)

Day 7 training: (484847, 891)
Day 7 validation: (105693, 891)


In [63]:
import numpy as np
from scipy.sparse import csr_matrix, hstack

interaction_train = csr_matrix(
    train_data["v258_m4_m5_interaction"].values.reshape(-1, 1),
    dtype=np.float32
)

interaction_validation = csr_matrix(
    validation_data["v258_m4_m5_interaction"].values.reshape(-1, 1),
    dtype=np.float32
)

print("Interaction training:", interaction_train.shape)
print("Interaction validation:", interaction_validation.shape)

Interaction training: (484847, 1)
Interaction validation: (105693, 1)


In [64]:
X_train_day8 = hstack([
    X_train_day7,
    interaction_train
]).tocsr()

X_validation_day8 = hstack([
    X_validation_day7,
    interaction_validation
]).tocsr()

print("Day 8 training:", X_train_day8.shape)
print("Day 8 validation:", X_validation_day8.shape)

Day 8 training: (484847, 892)
Day 8 validation: (105693, 892)


In [65]:
print("Training rows match:",
      X_train_day8.shape[0] == len(y_train))

print("Validation rows match:",
      X_validation_day8.shape[0] == len(y_validation))

print("New feature count:",
      X_train_day8.shape[1] - X_train_day7.shape[1])

Training rows match: True
Validation rows match: True
New feature count: 1


In [66]:
from xgboost import XGBClassifier

xgb_day8_exp1 = XGBClassifier(
    n_estimators=1500,
    max_depth=8,
    learning_rate=0.07,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

xgb_day8_exp1.fit(
    X_train_day8,
    y_train,
    eval_set=[
        (X_validation_day8, y_validation)
    ],
    verbose=100
)

print("Day 8 Experiment 1 trained successfully.")

[0]	validation_0-aucpr:0.31721
[100]	validation_0-aucpr:0.54090
[200]	validation_0-aucpr:0.56122
[300]	validation_0-aucpr:0.57337
[400]	validation_0-aucpr:0.58305
[500]	validation_0-aucpr:0.58899
[600]	validation_0-aucpr:0.59240
[700]	validation_0-aucpr:0.59406
[800]	validation_0-aucpr:0.59597
[900]	validation_0-aucpr:0.59648
[1000]	validation_0-aucpr:0.59867
[1100]	validation_0-aucpr:0.59899
[1200]	validation_0-aucpr:0.60034
[1300]	validation_0-aucpr:0.59898
[1400]	validation_0-aucpr:0.59929
[1499]	validation_0-aucpr:0.59811
Day 8 Experiment 1 trained successfully.


In [67]:
from sklearn.metrics import roc_auc_score, average_precision_score

day8_proba = xgb_day8_exp1.predict_proba(
    X_validation_day8
)[:, 1]

day8_roc_auc = roc_auc_score(
    y_validation,
    day8_proba
)

day8_pr_auc = average_precision_score(
    y_validation,
    day8_proba
)

print(f"Day 8 Experiment 1 ROC-AUC: {day8_roc_auc:.4f}")
print(f"Day 8 Experiment 1 PR-AUC:  {day8_pr_auc:.4f}")

Day 8 Experiment 1 ROC-AUC: 0.9206
Day 8 Experiment 1 PR-AUC:  0.5982


In [68]:
import joblib

joblib.dump(
    xgb_day8_exp1,
    "xgb_day8_exp1_rejected.pkl"
)

print("Day 8 Experiment 1 model saved.")

Day 8 Experiment 1 model saved.


In [69]:
np.save(
    "y_validation_proba_day8_exp1.npy",
    day8_proba
)

print("Day 8 predictions saved.")

Day 8 predictions saved.


In [ ]:
# Day 8 — Model Improvement Progress

## Day 8 Baseline
- Day 7 Champion ROC-AUC: 0.9219
- Day 7 Champion PR-AUC: 0.6031
- Original feature count: 891

## Day 8 Experiment 1 — Rejected
- Added feature: `v258_m4_m5_interaction`
- Interaction pattern:
  - V258 missing
  - M4 = M0
  - M5 = T
- New feature count: 892
- ROC-AUC: 0.9206
- PR-AUC: 0.5982
- Result: Rejected — both ROC-AUC and PR-AUC decreased.

## Key Finding
The V258/M4/M5 combination was strongly associated with hard false negatives, but adding it as a single interaction feature did not provide additional predictive value to XGBoost.

## Day 8 Status
- Experiment 1: Completed and rejected
- Day 7 Champion remains the best model.
- Do NOT retrain Experiment 1.
- Day 7 891-feature matrices remain the correct baseline.
- Day 8 892-feature experiment was only a controlled test and should not replace the baseline.

## Resume Point
Next: Day 8 Experiment 2.

Focus:
- Investigate broader hard-fraud patterns.
- Avoid random feature additions.
- Prioritize features showing meaningful differences between False Negatives and True Negatives.
- Compare every experiment against the Day 7 benchmark:
  - ROC-AUC: 0.9219
  - PR-AUC: 0.6031

In [1]:
print("train_data:", train_data.shape)
print("validation_data:", validation_data.shape)

print("X_train_day7:", X_train_day7.shape)
print("X_validation_day7:", X_validation_day7.shape)

print("Day 7 benchmark PR-AUC: 0.6031")

NameError: name 'train_data' is not defined

In [2]:
import numpy as np
import pandas as pd

from scipy.sparse import load_npz

# Load Day 7 baseline matrices
X_train_day7 = load_npz(
    "/Users/ayushkumar/Desktop/Fraudguard/data/processed/X_train.npz"
)

X_validation_day7 = load_npz(
    "/Users/ayushkumar/Desktop/Fraudguard/data/processed/X_validation.npz"
)

# Load targets
y_train = pd.read_csv(
    "/Users/ayushkumar/Desktop/Fraudguard/data/processed/y_train.csv"
).iloc[:, 0]

y_validation = pd.read_csv(
    "/Users/ayushkumar/Desktop/Fraudguard/data/processed/y_validation.csv"
).iloc[:, 0]

print("X_train_day7:", X_train_day7.shape)
print("X_validation_day7:", X_validation_day7.shape)
print("y_train:", y_train.shape)
print("y_validation:", y_validation.shape)

X_train_day7: (484847, 891)
X_validation_day7: (105693, 891)
y_train: (484847,)
y_validation: (105693,)


In [3]:
import pandas as pd

raw_path = "/Users/ayushkumar/Desktop/Fraudguard/data/raw/train_transaction.csv"

train_raw = pd.read_csv(raw_path)

print("Raw shape:", train_raw.shape)
print("Columns needed:", [
    col for col in ["TransactionID", "isFraud", "TransactionDT", "M4", "M5"]
    if col in train_raw.columns
])

Raw shape: (590540, 394)
Columns needed: ['TransactionID', 'isFraud', 'TransactionDT', 'M4', 'M5']


In [4]:
train_raw["transaction_day"] = (
    train_raw["TransactionDT"] // (24 * 60 * 60)
)

train_data_exp2 = train_raw[
    train_raw["transaction_day"] <= 145
].copy()

validation_data_exp2 = train_raw[
    train_raw["transaction_day"] > 145
].copy()

print("Training:", train_data_exp2.shape)
print("Validation:", validation_data_exp2.shape)

Training: (484847, 395)
Validation: (105693, 395)


In [5]:
print(
    "Training target alignment:",
    np.array_equal(
        train_data_exp2["isFraud"].values,
        y_train.values
    )
)

print(
    "Validation target alignment:",
    np.array_equal(
        validation_data_exp2["isFraud"].values,
        y_validation.values
    )
)

Training target alignment: True
Validation target alignment: True


In [6]:
train_data_exp2["m4_m5_missing"] = (
    train_data_exp2["M4"].isna() |
    train_data_exp2["M5"].isna()
).astype(np.float32)

validation_data_exp2["m4_m5_missing"] = (
    validation_data_exp2["M4"].isna() |
    validation_data_exp2["M5"].isna()
).astype(np.float32)

In [7]:
print("Training feature distribution:")
print(train_data_exp2["m4_m5_missing"].value_counts(normalize=True))

print("\nValidation feature distribution:")
print(validation_data_exp2["m4_m5_missing"].value_counts(normalize=True))

print("\nFN/TN signal will be evaluated next.")

Training feature distribution:
m4_m5_missing
1.0    0.599168
0.0    0.400832
Name: proportion, dtype: float64

Validation feature distribution:
m4_m5_missing
1.0    0.567502
0.0    0.432498
Name: proportion, dtype: float64

FN/TN signal will be evaluated next.


In [8]:
day7_proba = np.load(
    "/Users/ayushkumar/Desktop/Fraudguard/notebooks/y_validation_proba_day7_champion.npy"
)

print("Probabilities:", day7_proba.shape)

Probabilities: (105693,)


In [9]:
validation_check = validation_data_exp2.copy()

validation_check["probability"] = day7_proba

validation_check["prediction"] = (
    validation_check["probability"] >= 0.20
).astype(int)

validation_check["error_type"] = np.select(
    [
        (validation_check["isFraud"] == 1) &
        (validation_check["prediction"] == 0),

        (validation_check["isFraud"] == 1) &
        (validation_check["prediction"] == 1),

        (validation_check["isFraud"] == 0) &
        (validation_check["prediction"] == 1)
    ],
    [
        "False Negative",
        "True Positive",
        "False Positive"
    ],
    default="True Negative"
)

print(
    validation_check["error_type"].value_counts()
)

error_type
True Negative     101386
False Negative      1833
True Positive       1778
False Positive       696
Name: count, dtype: int64


In [10]:
for outcome in ["False Negative", "True Negative"]:
    
    subset = validation_check[
        validation_check["error_type"] == outcome
    ]
    
    print(
        f"{outcome}: "
        f"{subset['m4_m5_missing'].mean() * 100:.2f}%"
    )

False Negative: 46.81%
True Negative: 56.30%


In [11]:
train_data_exp2["dist1_missing"] = (
    train_data_exp2["dist1"].isna()
).astype(np.float32)

validation_data_exp2["dist1_missing"] = (
    validation_data_exp2["dist1"].isna()
).astype(np.float32)

print("Training dist1 missing:",
      train_data_exp2["dist1_missing"].mean() * 100)

print("Validation dist1 missing:",
      validation_data_exp2["dist1_missing"].mean() * 100)

Training dist1 missing: 60.582413
Validation dist1 missing: 55.385883


In [12]:
validation_check["dist1_missing"] = (
    validation_check["dist1"].isna()
).astype(np.float32)

for outcome in ["False Negative", "True Negative"]:
    subset = validation_check[
        validation_check["error_type"] == outcome
    ]

    print(
        f"{outcome}: "
        f"{subset['dist1_missing'].mean() * 100:.2f}%"
    )

False Negative: 66.94%
True Negative: 54.44%


In [15]:
# Recreate numerical feature list from the raw training data

TARGET = "isFraud"
ID_COLUMN = "TransactionID"

feature_columns = [
    col for col in train_data_exp2.columns
    if col not in [TARGET, ID_COLUMN, "transaction_day"]
]

categorical_features_exp2 = train_data_exp2[
    feature_columns
].select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_features_exp2 = [
    col for col in feature_columns
    if col not in categorical_features_exp2
]

# Add our Day 8 candidate
if "dist1_missing" not in numerical_features_exp2:
    numerical_features_exp2.append("dist1_missing")

print("Numerical features:", len(numerical_features_exp2))
print("Categorical features:", len(categorical_features_exp2))
print(
    "dist1_missing present:",
    "dist1_missing" in numerical_features_exp2
)

Numerical features: 380
Categorical features: 14
dist1_missing present: True


In [16]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack

# --------------------------------------------------
# 1. Numerical preprocessing
# --------------------------------------------------

numerical_imputer_exp2 = SimpleImputer(
    strategy="median"
)

X_train_num_exp2 = numerical_imputer_exp2.fit_transform(
    train_data_exp2[numerical_features_exp2]
)

X_validation_num_exp2 = numerical_imputer_exp2.transform(
    validation_data_exp2[numerical_features_exp2]
)

# --------------------------------------------------
# 2. Categorical preprocessing
# --------------------------------------------------

RARE_THRESHOLD = 50

categorical_maps_exp2 = {}

for feature in categorical_features_exp2:

    values = train_data_exp2[feature].fillna("__MISSING__")

    counts = values.value_counts()

    frequent_categories = counts[
        counts >= RARE_THRESHOLD
    ].index

    categorical_maps_exp2[feature] = set(
        frequent_categories
    )


def transform_categorical_exp2(
    df,
    categorical_columns,
    categorical_maps
):

    result = df[categorical_columns].copy()

    for feature in categorical_columns:

        result[feature] = result[feature].fillna(
            "__MISSING__"
        )

        frequent_categories = categorical_maps[feature]

        result[feature] = result[feature].where(
            result[feature].isin(frequent_categories),
            "__RARE__"
        )

    return result


X_train_cat_exp2 = transform_categorical_exp2(
    train_data_exp2,
    categorical_features_exp2,
    categorical_maps_exp2
)

X_validation_cat_exp2 = transform_categorical_exp2(
    validation_data_exp2,
    categorical_features_exp2,
    categorical_maps_exp2
)

# --------------------------------------------------
# 3. One-hot encoding
# --------------------------------------------------

categorical_encoder_exp2 = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True,
    dtype=np.float32
)

X_train_cat_encoded_exp2 = (
    categorical_encoder_exp2.fit_transform(
        X_train_cat_exp2
    )
)

X_validation_cat_encoded_exp2 = (
    categorical_encoder_exp2.transform(
        X_validation_cat_exp2
    )
)

# --------------------------------------------------
# 4. Final matrices
# --------------------------------------------------

X_train_exp2 = hstack([
    X_train_num_exp2,
    X_train_cat_encoded_exp2
]).tocsr()

X_validation_exp2 = hstack([
    X_validation_num_exp2,
    X_validation_cat_encoded_exp2
]).tocsr()

print("Experiment 2 training shape:", X_train_exp2.shape)
print("Experiment 2 validation shape:", X_validation_exp2.shape)

print("Training target:", y_train.shape)
print("Validation target:", y_validation.shape)

print(
    "Rows match:",
    X_train_exp2.shape[0] == len(y_train),
    X_validation_exp2.shape[0] == len(y_validation)
)

Experiment 2 training shape: (484847, 519)
Experiment 2 validation shape: (105693, 519)
Training target: (484847,)
Validation target: (105693,)
Rows match: True True


In [17]:
X_train_day7.shape
X_validation_day7.shape

(105693, 891)

In [18]:
from scipy.sparse import csr_matrix, hstack

# Create the candidate feature
dist1_missing_train = (
    train_data_exp2["dist1_missing"].values
    .reshape(-1, 1)
)

dist1_missing_validation = (
    validation_data_exp2["dist1_missing"].values
    .reshape(-1, 1)
)

# Append to the complete Day 7 representation
X_train_exp2 = hstack([
    X_train_day7,
    csr_matrix(dist1_missing_train)
]).tocsr()

X_validation_exp2 = hstack([
    X_validation_day7,
    csr_matrix(dist1_missing_validation)
]).tocsr()

print("Experiment 2 training shape:", X_train_exp2.shape)
print("Experiment 2 validation shape:", X_validation_exp2.shape)
print(
    "Rows match:",
    X_train_exp2.shape[0] == len(y_train),
    X_validation_exp2.shape[0] == len(y_validation)
)

Experiment 2 training shape: (484847, 892)
Experiment 2 validation shape: (105693, 892)
Rows match: True True


In [19]:
from xgboost import XGBClassifier

xgb_day8_exp2 = XGBClassifier(
    n_estimators=1500,
    max_depth=8,
    learning_rate=0.07,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

xgb_day8_exp2.fit(
    X_train_exp2,
    y_train,
    eval_set=[
        (X_validation_exp2, y_validation)
    ],
    verbose=100
)

print("Day 8 Experiment 2 trained successfully.")

[0]	validation_0-aucpr:0.31721
[100]	validation_0-aucpr:0.54077
[200]	validation_0-aucpr:0.56256
[300]	validation_0-aucpr:0.57541
[400]	validation_0-aucpr:0.58149
[500]	validation_0-aucpr:0.58734
[600]	validation_0-aucpr:0.59000
[700]	validation_0-aucpr:0.59367
[800]	validation_0-aucpr:0.59587
[900]	validation_0-aucpr:0.59745
[1000]	validation_0-aucpr:0.59809
[1100]	validation_0-aucpr:0.59757
[1200]	validation_0-aucpr:0.59997
[1300]	validation_0-aucpr:0.59978
[1400]	validation_0-aucpr:0.60002
[1499]	validation_0-aucpr:0.60036
Day 8 Experiment 2 trained successfully.


In [20]:
from sklearn.metrics import roc_auc_score, average_precision_score

day8_exp2_proba = xgb_day8_exp2.predict_proba(
    X_validation_exp2
)[:, 1]

roc_auc_exp2 = roc_auc_score(
    y_validation,
    day8_exp2_proba
)

pr_auc_exp2 = average_precision_score(
    y_validation,
    day8_exp2_proba
)

print(f"Day 8 Experiment 2 ROC-AUC: {roc_auc_exp2:.4f}")
print(f"Day 8 Experiment 2 PR-AUC:  {pr_auc_exp2:.4f}")

Day 8 Experiment 2 ROC-AUC: 0.9204
Day 8 Experiment 2 PR-AUC:  0.6004


In [21]:
# Day 8 — Experiment 3
# Distance-information missingness count

distance_features = ["dist1", "D2", "D5"]

train_data_exp2["dist_missing_count"] = (
    train_data_exp2[distance_features].isna().sum(axis=1)
).astype(np.float32)

validation_data_exp2["dist_missing_count"] = (
    validation_data_exp2[distance_features].isna().sum(axis=1)
).astype(np.float32)

print("Training distribution:")
print(train_data_exp2["dist_missing_count"].value_counts().sort_index())

print("\nValidation distribution:")
print(validation_data_exp2["dist_missing_count"].value_counts().sort_index())

Training distribution:
dist_missing_count
0.0     95184
1.0    134499
2.0    109828
3.0    145336
Name: count, dtype: int64

Validation distribution:
dist_missing_count
0.0    30287
1.0    28237
2.0    16998
3.0    30171
Name: count, dtype: int64


In [22]:
validation_check["dist_missing_count"] = (
    validation_check[distance_features].isna().sum(axis=1)
).astype(np.float32)

print("\nMean missing-count by outcome:")

for outcome in ["False Negative", "True Negative"]:
    subset = validation_check[
        validation_check["error_type"] == outcome
    ]

    print(
        f"{outcome}: "
        f"{subset['dist_missing_count'].mean():.3f}"
    )


Mean missing-count by outcome:
False Negative: 1.658
True Negative: 1.428


In [23]:
from scipy.sparse import hstack, csr_matrix

dist_count_train = (
    train_data_exp2["dist_missing_count"]
    .values
    .reshape(-1, 1)
)

dist_count_validation = (
    validation_data_exp2["dist_missing_count"]
    .values
    .reshape(-1, 1)
)

X_train_exp3 = hstack([
    X_train_day7,
    csr_matrix(dist_count_train)
]).tocsr()

X_validation_exp3 = hstack([
    X_validation_day7,
    csr_matrix(dist_count_validation)
]).tocsr()

print("Experiment 3 training shape:", X_train_exp3.shape)
print("Experiment 3 validation shape:", X_validation_exp3.shape)

print(
    "Rows match:",
    X_train_exp3.shape[0] == len(y_train),
    X_validation_exp3.shape[0] == len(y_validation)
)

Experiment 3 training shape: (484847, 892)
Experiment 3 validation shape: (105693, 892)
Rows match: True True


In [24]:
from xgboost import XGBClassifier

xgb_day8_exp3 = XGBClassifier(
    n_estimators=1500,
    max_depth=8,
    learning_rate=0.07,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

xgb_day8_exp3.fit(
    X_train_exp3,
    y_train,
    eval_set=[
        (X_validation_exp3, y_validation)
    ],
    verbose=100
)

print("Day 8 Experiment 3 trained successfully.")

[0]	validation_0-aucpr:0.31721
[100]	validation_0-aucpr:0.53747
[200]	validation_0-aucpr:0.56194
[300]	validation_0-aucpr:0.57376
[400]	validation_0-aucpr:0.58100
[500]	validation_0-aucpr:0.58600
[600]	validation_0-aucpr:0.58916
[700]	validation_0-aucpr:0.59250
[800]	validation_0-aucpr:0.59560
[900]	validation_0-aucpr:0.59538
[1000]	validation_0-aucpr:0.59614
[1100]	validation_0-aucpr:0.59704
[1200]	validation_0-aucpr:0.59903
[1300]	validation_0-aucpr:0.59880
[1400]	validation_0-aucpr:0.59958
[1499]	validation_0-aucpr:0.59977
Day 8 Experiment 3 trained successfully.


In [25]:
from sklearn.metrics import roc_auc_score, average_precision_score

day8_exp3_proba = xgb_day8_exp3.predict_proba(
    X_validation_exp3
)[:, 1]

roc_auc_exp3 = roc_auc_score(
    y_validation,
    day8_exp3_proba
)

pr_auc_exp3 = average_precision_score(
    y_validation,
    day8_exp3_proba
)

print(f"Day 8 Experiment 3 ROC-AUC: {roc_auc_exp3:.4f}")
print(f"Day 8 Experiment 3 PR-AUC:  {pr_auc_exp3:.4f}") 

Day 8 Experiment 3 ROC-AUC: 0.9187
Day 8 Experiment 3 PR-AUC:  0.5998


In [26]:
import joblib
import numpy as np

joblib.dump(
    xgb_day8_exp3,
    "xgb_day8_exp3_rejected.pkl"
)

np.save(
    "y_validation_proba_day8_exp3.npy",
    day8_exp3_proba
)

print("Day 8 Experiment 3 model saved.")
print("Day 8 Experiment 3 predictions saved.")

Day 8 Experiment 3 model saved.
Day 8 Experiment 3 predictions saved.
